# ShelfGuard Discount Model Training

This notebook trains and evaluates the final LightGBM discount prediction model.
The target `discount_pct` is stored as a fraction: `0.36` means a `36%` discount.
The saved model uses the same five feature names consumed by `pricing_model.py`.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "perishable_goods_management.csv"
MODEL_PATH = PROJECT_DIR / "pricing_model.pkl"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH.name}")

Loaded 100,000 rows from perishable_goods_management.csv


In [2]:
required_columns = {
    "transaction_date",
    "expiration_date",
    "shelf_life_days",
    "initial_quantity",
    "supplier_score",
    "is_promoted",
    "discount_pct",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["expiration_date"] = pd.to_datetime(df["expiration_date"])
df["days_to_expiry"] = (
    df["expiration_date"] - df["transaction_date"]
).dt.days
df["stock_level"] = df["initial_quantity"]
df["remaining_shelf_life_pct"] = (
    df["days_to_expiry"] / df["shelf_life_days"]
) * 100

features = [
    "days_to_expiry",
    "stock_level",
    "remaining_shelf_life_pct",
    "supplier_score",
    "is_promoted",
]
X = df[features]
y = df["discount_pct"]

if X.isnull().any().any() or y.isnull().any():
    raise ValueError("Training features and target must not contain missing values")

print(f"Features: {features}")
print(f"Target range: {y.min():.2f} to {y.max():.2f} (fraction)")
print(f"Target range: {y.min() * 100:.2f}% to {y.max() * 100:.2f}%")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

base_model = LGBMRegressor(
    objective="regression",
    verbosity=-1,
    n_jobs=-1,
    random_state=42,
)
param_grid = {
    "n_estimators": [200, 400],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [15, 31],
    "max_depth": [-1, 5],
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

final_model = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")
predictions = final_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Holdout MAE: {mae:.4f} (fraction)")
print(f"Holdout RMSE: {rmse:.4f} (fraction)")
print(f"Holdout R2: {r2:.4f}")

from sklearn.model_selection import RepeatedKFold, cross_validate

repeated_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
cv_results = cross_validate(
    final_model,
    X,
    y,
    cv=repeated_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
    n_jobs=-1,
)

cv_mae = -cv_results["test_mae"]
cv_rmse = -cv_results["test_rmse"]
cv_r2 = cv_results["test_r2"]
print("\nRepeated 5-fold cross-validation (3 repeats):")
print(f"MAE: {cv_mae.mean():.4f} +/- {cv_mae.std():.4f} (fraction)")
print(f"RMSE: {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f} (fraction)")
print(f"R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")

feature_importance = pd.Series(
    final_model.feature_importances_,
    index=features,
).sort_values(ascending=False)
print("\nFeature importance:")
print(feature_importance)

joblib.dump(final_model, MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

loaded_model = joblib.load(MODEL_PATH)
loaded_predictions = loaded_model.predict(X_test.head(5))
print("Reload verification:", loaded_predictions)

Features: ['days_to_expiry', 'stock_level', 'remaining_shelf_life_pct', 'supplier_score', 'is_promoted']
Target range: 0.00 to 0.75 (fraction)
Target range: 0.00% to 75.00%
Training rows: 80,000
Test rows: 20,000
Best parameters: {'learning_rate': 0.03, 'max_depth': 5, 'n_estimators': 200, 'num_leaves': 15}
Holdout MAE: 0.0951 (fraction)
Holdout RMSE: 0.1343 (fraction)
Holdout R2: 0.4390

Repeated 5-fold cross-validation (3 repeats):
MAE: 0.0950 +/- 0.0005 (fraction)
RMSE: 0.1342 +/- 0.0006 (fraction)
R2: 0.4410 +/- 0.0072

Feature importance:
stock_level                 1246
days_to_expiry               881
remaining_shelf_life_pct     392
supplier_score               235
is_promoted                   46
dtype: int32
Saved model to c:\Users\Mypc\Documents\AICW\ShelfGuard\pricing_model.pkl
Reload verification: [0.02855396 0.1552672  0.07668578 0.21108886 0.02977638]
